# Qwen3.5-4B anchor length sweep (single-run Colab)

Этот notebook делает один дорогой прогон и сравнивает **short / medium / long** anchor spans на одном и том же наборе кейсов.

Он сохраняет:
- JSON с `curve_points` и полными результатами по каждому профилю;
- Markdown report с таблицей `anchor length -> quality`.


In [ ]:
!nvidia-smi || true
!python --version
!pip install -q --upgrade pip
!pip install -q torch torchvision pillow accelerate einops pytest numpy nbformat "transformers @ git+https://github.com/huggingface/transformers.git@main"


In [ ]:
%cd /content
import os
import subprocess
from pathlib import Path

if not os.path.exists('/content/ABPT'):
    !git clone https://github.com/kharkilirov1/Anchor-engine.git ABPT
else:
    %cd /content/ABPT
    !git pull --ff-only

%cd /content/ABPT
print('HEAD =', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
import json
from pathlib import Path

MODEL_NAME = 'Qwen/Qwen3.5-4B'
DEVICE = 'cuda'
MAX_LENGTH = 160
MAX_NEW_TOKENS = 120
PROFILES = ['short', 'medium', 'long']
SWEEP_JSON = '/content/ABPT/archive/qwen35_4b_anchor_length_sweep.json'
SWEEP_MD = '/content/ABPT/docs/research/qwen35_4b_anchor_length_sweep.md'

for path in [SWEEP_JSON, SWEEP_MD]:
    Path(path).parent.mkdir(parents=True, exist_ok=True)

print('MODEL_NAME =', MODEL_NAME)
print('PROFILES =', PROFILES)


## Main run

Этот шаг прогоняет все три профиля длины anchor-а на одном и том же model host.


In [ ]:
import subprocess
cmd = [
    'python', 'scripts/run_qwen_anchor_length_sweep.py',
    '--model', MODEL_NAME,
    '--device', DEVICE,
    '--max_length', str(MAX_LENGTH),
    '--max_new_tokens', str(MAX_NEW_TOKENS),
    '--profiles', *PROFILES,
    '--output_json', SWEEP_JSON,
    '--output_md', SWEEP_MD,
]
print('RUN:', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
if result.stdout:
    print(result.stdout)
if result.returncode != 0:
    if result.stderr:
        print('STDERR:')
        print(result.stderr)
    raise RuntimeError(f'anchor length sweep failed with code {result.returncode}')


In [ ]:
sweep = json.loads(Path(SWEEP_JSON).read_text(encoding='utf-8'))
print('metadata:')
print(json.dumps(sweep['metadata'], indent=2, ensure_ascii=False))
print()
print('curve points:')
print(json.dumps(sweep['curve_points'], indent=2, ensure_ascii=False))
print()
print('best profile:')
print(json.dumps(sweep['best_profile'], indent=2, ensure_ascii=False))
print()
print('report head:')
print(Path(SWEEP_MD).read_text(encoding='utf-8')[:7000])
